# Решения · ML Handbook гл. 45 (training tricks)

Открывай **только после** [handbook_gl45_практика.ipynb](./handbook_gl45_практика.ipynb).

Источник: [training-tricks.html](https://mlhandbook-production.up.railway.app/parts/deep-learning/training-tricks.html)

Теория: [../d.5.45 Тонкости обучения нейросетей.md](../d.5.45 Тонкости обучения нейросетей.md) · [README.md](./README.md)

## Зачем практику handbook

Закрепить главу **руками**: перед каждым блоком — **Зачем этот фрагмент** и ссылка на конспект; после запуска — **Что проверить** по `print`.

| ☐ | Блок |
|---|------|
| ☐ | §45.1 He |
| ☐ | §45.2 BN vs LN |
| ☐ | §45.3 clip |


## Импорты

Один раз в начале.


In [ ]:
import torch
import torch.nn as nn


# §45.1 · He init

## Зачем этот фрагмент

Kaiming/He под ReLU: дисперсия активаций не взрывается на старте. Повтор [d.4.36](../../../d.4%20Deep%20Learning/d.4.36%20Оптимизация%20и%20обучение%20в%20DL/d.4.36%20Оптимизация%20и%20обучение%20в%20DL.md) на уровне одного слоя.

См. [d.5.45 §2](../d.5.45 Тонкости обучения нейросетей.md#2-инициализация-handbook-451).

## Что проверить

`std` весов близок к **sqrt(2/fan_in)** ≈ 0.088 при fan_in=256.


In [ ]:
layer = nn.Linear(256, 128)
nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
print('std', round(layer.weight.std().item(),3), 'target', round((2/256)**0.5, 3))


---


# §45.2 · BN vs LN

## Зачем этот фрагмент

LayerNorm нормирует **по признакам одного объекта** — как в трансформерах; не путать с BatchNorm по батчу.

См. [d.5.45 §3](../d.5.45 Тонкости обучения нейросетей.md#3-нормализации-handbook-452).

## Что проверить

После LN mean ≈ 0, std ≈ 1 **по всему тензору** (для одного forward).


In [ ]:
x = torch.randn(16, 64)
ln = nn.LayerNorm(64)(x)
print('LN mean', round(ln.mean().item(),3), 'std', round(ln.std().item(),2))


---


# §45.3 · clip_grad_norm

## Зачем этот фрагмент

Если lr велик или последовательность длинная — норма градиента режется **перед** `optimizer.step()`.

См. [d.5.45 §4](../d.5.45 Тонкости обучения нейросетей.md#4-clip-градиента-handbook-453).

## Что проверить

В print — норма **до** clipping (может быть > max_norm).


In [ ]:
model = nn.Sequential(nn.Linear(64,64), nn.ReLU(), nn.Linear(64,10))
model(torch.randn(8,64)).sum().backward()
n = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
print('norm was', round(n.item(), 1))
